### 🔰PyTorchでニューラルネットワーク基礎 #34 DataLoader編


### 内容
* Qiitaの記事と連動しています
* 各種ファイルの保存先は環境によって適宜変更してください


### データについて
* ramen_data.csv: 架空のデータ（表タイプ、分類問題）
* history_text_label_id.jsonl: wikipediaの日本の歴史分野の内容から収集した時代区分を判定するテキスト
    * ids列の長さがそれぞれことなる。
    * 時代区分を直接表すテキストは除いてある。（江戸時代ラベルの文章に「江戸」という単語がはありません）
    * クラス数：５（弥生、奈良、室町、江戸、昭和）

### カスタム版Datasetクラス
* [第33回](https://qiita.com/AzukiImo/items/1a051a5c6f1d0296f4ed)に掲載

### DataLoaderの基本
ライブラリーの読み込み

In [1]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

### 1.2 TensorDatasetと組み合わせてみた

In [2]:
# ラーメン分類CSVファイル
filename = "./data/ramen_data.csv" 
x = np.loadtxt(filename, delimiter=",", skiprows=1, usecols=(0,1,2,3,4))
t = np.loadtxt(filename, delimiter=",", skiprows=1, usecols=(6))
x = torch.FloatTensor(x)
t = torch.LongTensor(t)

dataset = TensorDataset(x,t)

dataloader = DataLoader(
    dataset = dataset,
    batch_size = 4,
    shuffle = True
)

for x, t in dataloader:
    # dataに対する処理
    print(f"{x=}")
    print(f"{t=}")
    break

x=tensor([[ 8.8000, 10.0000,  7.8000,  5.6000,  2.0000],
        [ 3.7000,  4.7000,  4.8000,  2.0000,  6.4000],
        [ 7.2000,  9.1000,  6.0000,  6.5000,  1.7000],
        [ 2.4000,  4.6000,  6.6000,  1.5000,  8.3000]])
t=tensor([1, 0, 1, 0])


### 1.3 カスタム版Datasetと組み合わせてみた
* MyDataset02：numpy配列の辞書が出力値になるように設計

In [3]:
class MyDataset02(Dataset):
    # (1) 引数が２種類(x, t)
    def __init__(self, x,t):
        self.x = x   # 入力データ
        self.t = t   # 教師データ
   
   # (2) どちらかのサイズでOK
    def __len__(self):
        return len(self.t)
    
    # (3) 入出力するデータのタイプによって適宜修正
    def __getitem__(self, index):
        return {
            "data":  np.array(self.x[index], dtype=np.float32),
            "label": np.array(self.t[index], dtype=np.int64)
            }

filename = "./data/ramen_data.csv" 
x = np.loadtxt(filename, delimiter=",", skiprows=1, usecols=(0,1,2,3,4))
t = np.loadtxt(filename, delimiter=",", skiprows=1, usecols=(6))

dataset = MyDataset02(x,t)

dataloader = DataLoader(
    dataset = dataset,
    batch_size = 4,
    shuffle = True
)

for data in dataloader:
    # dataに対する処理
    print(f"{data=}")
    break

data={'data': tensor([[8.7000, 9.6000, 7.2000, 6.4000, 1.8000],
        [2.2000, 4.3000, 4.5000, 1.8000, 5.3000],
        [3.7000, 5.6000, 8.7000, 4.4000, 8.2000],
        [3.0000, 3.9000, 5.9000, 2.1000, 6.5000]]), 'label': tensor([1, 0, 2, 0])}


### 1.4 エラー
* history_text_id.jsonlを利用
* 系列長が異なるデータに対して、デフォルトcollate_fnをそのままつかうとエラー

In [4]:
class MyDataset03(Dataset):
    # (1) dataがデータフレームになっている
    def __init__(self, data):
        self.data = data
   
    def __len__(self):
        return len(self.data)
    
    # (2) データの形式で適宜修正
    def __getitem__(self, index):
        item = self.data.iloc[index]   # データフレームのindex行を取得したいので data.iloc[]を使う
        tensor_x = torch.tensor(item["ids"], dtype=torch.long)
        tensor_t = torch.tensor(item["label"], dtype=torch.long)
        return {"ids": tensor_x, "label": tensor_t}

        
data_filename = "./data/history_text_label_id.jsonl"       # 分類問題のデータ
df = pd.read_json(data_filename, lines=True)

dataset = MyDataset03(df)
print(f"{dataset[0]=}")

dataloader = DataLoader(
    dataset = dataset,
    batch_size = 4,
)

# 次のプリント文はエラーになる
# print(next(iter(dataloader)))
# RuntimeError: stack expects each tensor to be equal size, but got [12] at entry 0 and [22] at entry 1 

dataset[0]={'ids': tensor([  1,  29,  90,   5,  46,   8,  88,  97,   5, 105,   6,   2]), 'label': tensor(0)}


## 2. collate_fnの作成
### 2.2 tensor値を持つ辞書形式に変換するcollate_fn
* ramen_data.csvをつかう

In [5]:
def sample_collate_fn(batch):
    # (1) batch[0]はdataとlabelをキーに持つnumpy配列
    data = torch.as_tensor(np.stack([x["data"] for x in batch]), dtype=torch.float)
    labels = torch.as_tensor(np.stack([x["label"] for x in batch]), dtype=torch.long)
    
    return {"data": data, "label": labels}

dataset = MyDataset02(x,t)
dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=sample_collate_fn,
)
next(iter(dataloader))

{'data': tensor([[ 6.9000,  9.7000,  6.5000, 10.0000,  2.1000],
         [ 4.3000,  5.8000,  9.1000,  4.3000,  6.7000],
         [ 2.6000,  2.7000,  4.5000,  2.5000,  6.3000],
         [ 3.5000,  5.3000, 10.0000,  4.0000,  5.7000]]),
 'label': tensor([1, 2, 0, 2])}

### 2.3 異なる系列長のデータをミニバッチごとに揃えるcollate_fn
* history_text_label_id.jsonlをつかう

In [6]:
from torch.nn.utils.rnn import pad_sequence # 等長化のときに利用する

def padding_collate_fn(batch):
    # (1) 入力されるデータをtensorに変換
    ids_list = [x["ids"] for x in batch]
    labels = torch.stack([x["label"] for x in batch])
    # (2) pad_id = 0としています
    padded_ids = pad_sequence(ids_list, batch_first=True, padding_value=0)
    # (3) padマスクの作成
    attention_mask = (padded_ids != 0).long()  # 実トークン=1, <pad>=0
    # (4)
    return {"ids": padded_ids, "attention_mask": attention_mask, "label": labels}


dataset = MyDataset03(df)

dataloader = DataLoader(
    dataset = dataset, 
    batch_size=4,
    shuffle = True,
    collate_fn=padding_collate_fn,
    )

torch.set_printoptions(linewidth=200,  profile="full")  # 省略なしで全部表示
next(iter(dataloader))

{'ids': tensor([[   1,  433,  579,  186, 1869,   55,    5,   56,   15, 1909,   55,   10,  349,    7, 1762,   30,   17,   29,  309,  582,   75,   20,    8,   37, 1909,   12,  201, 1988,   30,   59,  246,  346,
            76,   14,   68,  258,   26,  299,   54,    9,  213,   22,   21,  106, 1256,    9,  579,  505, 1973,    7,  701,   41,    6,    2],
         [   1,  946,   13,  834,   16,  510,    5,    3, 1406,   11,  323,   24,   12,  166,   32,    3,    5,    3, 1406,   11, 1991,   82,   14, 1867,  517,  541,    6,    2,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
         [   1,  650,  250,   43,  131,  262,    9, 1008,    3, 1129,    3, 1129,   76,  963,    5, 1853,  111, 1329,  134,   10,  159,  656,  461,    6,    2,    0,    0,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,  

### 2.4 クラスを使って表現してみた
* collate_fnに直接記述するパターン

In [7]:
class PaddingCollator:
    """
    padding_collate_fn をクラスで表現したバージョン
    """
    # (1)
    def __init__(self, pad_id=0):
        self.pad_id = pad_id

    # (2) padding_collate_fnと同じ
    def __call__(self, batch):
        ids_list = [x["ids"] for x in batch]
        labels = torch.stack([x["label"] for x in batch])
        padded_ids = pad_sequence(ids_list, batch_first=True, padding_value=self.pad_id)
        attention_mask = (padded_ids != self.pad_id).long()  # 実トークン=1, <pad>=0
        return {"ids": padded_ids, "attention_mask": attention_mask, "label": labels}

# (3) __call__の特徴を利用して、collatorを関数として機能させる
collator = PaddingCollator(pad_id=0)

dataset = MyDataset03(df)

# (4) インスタンスがそのまま callable なので関数のように記述してもOK
dataloader = DataLoader(
    dataset = dataset, 
    batch_size=4,
    shuffle = True,
    collate_fn=PaddingCollator(pad_id=0), # 直接書いてもOK
    )

print(next(iter(dataloader)))

{'ids': tensor([[   1,   89,   37,  420,   10,   98,   12, 1770, 1827, 1104,   32,   89,   37,  269,   10, 1979,  112, 1770, 1827,    7,  273,  235,   18,    6,    2,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
        [   1, 1091,  825,    3, 1393,    3, 1651,    3, 1072,  531,   36,   29,  993, 1287, 1288,   16,    3,  843,    5, 1801,  592,  309, 1934,    8,    3,  232,   15,   89,   15,  159,  370,   38,
          194,  136,  118,  161,    5,    3,  592,  309, 1934,   38,   14,  137,  213,  461,    6,    2],
        [   1,   65,   12,  964,  773,  165,    6,    2,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
        [   1,  650,  250,   68,  262,    9,   36, 1380,    3, 1048,  6

### PaddingCollatorのインスタンスを作成して、collate_fnにつかうパターン

In [8]:
dataset = MyDataset03(df)

collator = PaddingCollator(pad_id=0)

dataloader = DataLoader(
    dataset = dataset, 
    batch_size=2,
    shuffle = True,
    collate_fn=collator,
    )

torch.set_printoptions(linewidth=200,  profile="full")  # 省略なしで全部表示
next(iter(dataloader))

{'ids': tensor([[   1,    3,  437,    5,  760,  111,  115,    8, 1441,  201,    5,  967,  114,   10, 1587,   22,   75, 1119,  669,    3,  293,   45,  951,   36,  481,  250,  650,  262,    5,  433,   95,  111,
           409,  279,  700,  782,  497,  143, 1380,    3, 1048,  613,  308,   10,  126,  780,  352,  129,   64,    5,  323,  363,  963,  671,  815,    7,  667, 1742,   12,  203,  180,    8,   67,  261],
         [   1,  259,  491,  261,   36,  561,   61,    5,   61, 1273,    7,    3, 1544,   19,   38, 1416,   45,    3,    9,  145,   12,   93,    8,   23, 1956,   16,    3, 1956,    5, 1477,  803,    7,
           190,  313,  550,   34,    8,  161,  162,    5,    3,   90,    7,   11,   62,   41,    6,    2,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 